# Dataset Alignment Walkthrough

This generated notebook is the recipe companion for
`examples/api_dataset_alignment_demo.py`.

Start from channels sampled on different depth grids, align them to a shared index, convert depth units, and render the normalized result.

What you will practice in this walkthrough:

- Sort descending or irregular input indices into the order expected by the renderer.
- Reindex curves and rasters onto a shared sampling grid before layout binding.
- Convert depth units at the dataset stage instead of hard-coding unit changes later.
- Render a simple report once the channels share the same depth basis.

Prerequisites:

- `pip install wellplot`
- run the notebook from a checkout of this repository so the `examples/` files and sample data are available

Runtime model:

- import `wellplot` from the active installed environment
- use the repository checkout for the example files, helper modules, and sample data


In [ ]:
import sys
from pathlib import Path

try:
    import wellplot
except ImportError as exc:
    raise RuntimeError(
        "Install the published 'wellplot' package in the active "
        "environment before running this notebook."
    ) from exc

# Walk upward from the current working directory until we find the
# repository checkout that holds the example sources and sample data.
cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "examples").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "Run this notebook from a checkout of the wellplot repository "
        "so the example files and sample data are available."
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

examples_path = str(EXAMPLES_DIR)
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

print("wellplot version:", wellplot.__version__)
print("Examples root:", EXAMPLES_DIR)
print("Render output:", WORKSPACE_RENDERS)


In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "api_dataset_alignment_demo.py"
display(Code(source_path.read_text(), language="python"))


In [ ]:
# Import the example helpers so the notebook stays close to the script version.
import api_dataset_alignment_demo as demo

# Build the aligned dataset and inspect the normalized channels.
dataset = demo.build_aligned_dataset()
print("Channels:", sorted(dataset.channels))
for mnemonic in ("GR", "CBL", "VDL_SYN"):
    channel = dataset.get_channel(mnemonic)
    print(
        f"{mnemonic}: depth_unit={channel.depth_unit}, "
        f"samples={channel.depth.size}"
    )


In [ ]:
# Build the report that consumes the aligned dataset and render it to the
# same workspace output used by the example script.
from wellplot import render_report

report = demo.build_report(dataset)
output_path = WORKSPACE_RENDERS / "api_dataset_alignment_demo.pdf"
result = render_report(report, output_path=output_path)
print("Pages:", result.page_count)
print("Rendered:", result.output_path)


## Adapt Dataset Alignment Walkthrough Safely

- Use `sort_index(...)` first whenever incoming data arrives in descending acquisition order.
- Use `reindex_to(...)` early if later calculations assume one common depth axis.
- Convert the dataset index unit once and keep the report layout in that same unit to avoid accidental mixed-unit plots.
